# Creating a Semantic Relationships Graph from an IFC file

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed TopologicPy Classes

In [ ]:
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Set default mappings (do not change key names)

In [ ]:
# Do not change the code below this line
rels_color_mapping = {"IfcRelConnectsPathElements": "#440154",
                      "IfcRelContainedInSpatialStructure": "#31688E",
                      "IfcRelFillsElement": "#35B779",
                      "IfcRelSpaceBoundary": "#FDE725",
                      "IfcRelVoidsElement": "#E64B5D"
                      }

 
ifc_color_mapping = {"ifcbeam":"#440154",
                     "ifccovering": "#482878",
                     "ifcdoor": "#3E4989",
                     "ifcfooting": "#31688E",
                     "ifcfurnishingelement": "#26828E",
                     "ifcmember":"#1F9E89",
                     "ifcopeningelement":"#35B779",
                     "ifcrailing":"#6DCD59",
                     "ifcslab":"#B4DE2C",
                     "ifcspace":"#FDE725",
                     "ifcstairflight":"#FCA636",
                     "ifcwall":"#F8765C",
                     "ifcwallstandardcase":"#E64B5D",
                     "ifcwindow":"#D41159"
                     }


## 5. Specify the path to the IFC file and what to include

In [ ]:
# Choose your own IFC file if you wish
ifc_file_path = r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\Ifc2x3_Duplex_Architecture.ifc"

## 6. Import the IFC file as graph

In [ ]:
graph = Graph.ByIFCPath(ifc_file_path,
                        transferDictionaries=True,
                        storeBREP = True,
                        mode="geometry",
                       )


## 7. Extract breps from the vertices and set colours for vertices and edges

In [ ]:
vertices = Graph.Vertices(graph)
boxes = []
for v in vertices:
    d = Topology.Dictionary(v)
    brep = Dictionary.ValueAtKey(d, "BREP")
    topology = Topology.ByBREPString(brep)
    box = Topology.BoundingBox(topology)
    boxes.append(box)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "Unknown")
    vertexColor = ifc_color_mapping.get(ifc_type, "white")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, vertexColor])
    v = Topology.SetDictionary(v, d)

edges = Graph.Edges(graph)
rels = []
for e in edges:
    d = Topology.Dictionary(e)
    ifc_rel = Dictionary.ValueAtKey(d, "IFC_type")
    edgeColor = rels_color_mapping.get(ifc_rel, "white")
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [3, edgeColor])
    e = Topology.SetDictionary(e, d)

## 8. Show the final result

In [ ]:
Topology.Show(boxes, graph,
              faceOpacity=0.1,
              sagitta=0.15,
              absolute=False,
              backgroundColor="black",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              width=800,
              height=600,
              renderer=renderer)

## 9. Use Pyvis for an alternative visualisation

In [ ]:
pyvis_graph = Graph.PyvisGraph(graph, path=r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\pyvis_graph.html",
                               vertexSizeKey="size",
                               vertexColorKey="color",
                               vertexLabelKey="IFC_type",
                               edgeWeightKey="width",
                               edgeColorKey="color")